# Incomes analysis

Third project alongside `../expenses/` and `../liabilities/`, same two-layer architecture,
applied to the snapshot's income sources: IRS declarations (annual, gross, household),
NetSalary (monthly, net, proponent), Social Security payments, hAPI `CalculatedIncomes`,
self-employment activity and invoicing, pensions.

- **Layer 1** `income_features(profile)`: full versioned feature vector.
- **Layer 2** `display_profile` / `render_profile_text` (pt-PT): the human-readable view.

Module: `incomes_features.py` (stdlib only). Output: `incomes_layers_output.csv`.

Design decisions:
1. Sources are reported **side by side, never mixed silently**: IRS is gross, annual and
   household-wide (couples declare together); salary is net, monthly, proponent-only.
   `salary_vs_irs_ratio` makes the cross-source comparison explicit instead of hiding it.
2. Monthly-series entries with `Month = 0` (annual aggregates / bonus rows) are excluded
   from monthly averages.
3. Sub-1-EUR annual IRS declarations count as missing (same rule as expenses).
4. "6m" averages cover the last 6 recorded months of that source.

In [ ]:
import json
import sys

import pandas as pd

sys.path.insert(0, ".")
from incomes_features import display_profile, income_features, render_profile_text

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 60)

raw = pd.read_csv("../data/data_full.csv")
snap = raw.drop_duplicates("uuid")[["uuid", "value"]].reset_index(drop=True)
print(f"{len(snap)} proponents")

## 1. Layer 1: build the feature vector for every proponent

In [ ]:
feats = pd.DataFrame([
    {"uuid": u, **income_features(json.loads(v))}
    for u, v in zip(snap["uuid"], snap["value"])
])
print(f"feats: {feats.shape[0]} proponents x {feats.shape[1] - 1} fields, "
      f"version {feats['features_version'].iloc[0]}")

KEY = ["irs_total_income", "irs_yoy_growth", "avg_net_salary_6m",
       "avg_gross_salary_6m", "variable_income_share_6m", "avg_regular_income_6m",
       "salary_vs_irs_ratio", "n_income_sources"]
print("\npopulation overview:")
print(feats[KEY].describe(percentiles=[.25, .5, .75, .9]).round(2).T.to_string())

print("\ncoverage: salary", int(feats["has_salary_data"].sum()),
      "| ss", int(feats["has_ss_data"].sum()),
      "| activity open", int((feats["activity_open"] == 1).sum()),
      "| invoices", int(feats["has_invoices"].sum()),
      "| pension", int(feats["has_pension"].sum()),
      "| joint IRS", int((feats["irs_is_joint"] == 1).sum()),
      "| zero sources", int((feats["n_income_sources"] == 0).sum()))

In [ ]:
import matplotlib.pyplot as plt

INK, INK2, BLUE = "#0b0b0b", "#52514e", "#2a78d6"

fig, axes = plt.subplots(1, 2, figsize=(11.5, 4))
d = feats["irs_total_income"].dropna()
axes[0].hist(d[d > 0], bins=40, color=BLUE, edgecolor="white", linewidth=0.8)
axes[0].set_xscale("log")
axes[0].set_title("IRS annual income (EUR, log)", loc="left", color=INK, fontsize=10)

r = feats["salary_vs_irs_ratio"].dropna()
axes[1].hist(r[r < 3], bins=30, color=BLUE, edgecolor="white", linewidth=0.8)
axes[1].axvline(1.0, color=INK2, ls="--", lw=1)
axes[1].set_title("Gross salary annualized / IRS (1.0 = coherent)", loc="left", color=INK, fontsize=10)

for ax in axes:
    ax.tick_params(colors=INK2, labelsize=9)
    ax.grid(axis="y", color="#e6e5e0", lw=0.6)
    ax.set_axisbelow(True)
    for s in ("top", "right"):
        ax.spines[s].set_visible(False)
    for s in ("left", "bottom"):
        ax.spines[s].set_color("#c3c2b7")
plt.tight_layout()
plt.show()

## 2. Layer 2: the human-readable profile

Representative cases: salaried, IRS-only, self-employed with invoicing, and no verifiable income.

In [ ]:
fset = feats.set_index("uuid")
vmap = dict(zip(snap["uuid"], snap["value"]))
examples = [
    fset[(fset["has_salary_data"] == 1) & (fset["variable_income_share_6m"] > 0.2)].index[0],
    fset[(fset["has_salary_data"] == 0) & fset["irs_total_income"].notna()].index[0],
    fset[(fset["activity_open"] == 1) & (fset["invoices_total_12m"] > 0)].index[0],
    fset[fset["n_income_sources"] == 0].index[0],
]
for u in examples:
    print(f"=== {u[:8]} ===")
    print(render_profile_text(income_features(json.loads(vmap[u]))))
    print()

## 3. Validation: module vs independent pandas computation

Core figures recomputed independently. Tolerance 0.015 EUR absorbs banker's-rounding
ties on averages. Must print 0.

In [ ]:
mism = checked = 0
for uuid, v in zip(snap["uuid"], snap["value"]):
    p = json.loads(v)
    vec = income_features(p)
    checks = {}
    ns = pd.DataFrame(p.get("NetSalary") or [])
    if not ns.empty:
        ns = ns[(ns["Year"].fillna(0) > 0) & (ns["Month"].fillna(0) > 0)].sort_values(["Year", "Month"])
        if len(ns):
            checks["avg_net_salary_6m"] = ns.tail(6)["NetSalary"].mean()
            checks["salary_months_covered"] = len(ns)
    ss = pd.DataFrame((p.get("SSPayments") or {}).get("Payments") or [])
    if not ss.empty:
        ss = ss[(ss["Year"].fillna(0) > 0) & (ss["Month"].fillna(0) > 0)].sort_values(["Year", "Month"])
        if len(ss):
            checks["avg_ss_amount_6m"] = ss.tail(6)["Amount"].mean()
    inv = pd.DataFrame(p.get("InvoicesList") or [])
    if not inv.empty:
        checks["invoices_total_12m"] = (inv.sort_values(["Year", "Month"])
                                        .tail(12)["TotalAmount"].fillna(0).sum())
    for k, expect in checks.items():
        checked += 1
        got = vec[k]
        if got is None or abs(float(got) - float(expect)) > 0.015:
            mism += 1
            print("MISMATCH", uuid[:8], k, got, round(expect, 2))
print(f"{checked} comparisons, mismatches: {mism}")